In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import numpy as np
import os
import pandas as pd
from sklearn.metrics import ndcg_score, roc_auc_score
from pyspark.sql import DataFrame

In [0]:
full_results = spark.sql('select * from ds_sandbox.next_uk_nextAds_predict_prod_half')

In [0]:
high_rep_ratio_themes_sc = ['homesofas','homediningroom','homecurtainsandblinds','mensouterwear','womensouterwear','reisscoatsjacketswomens','reisscoatsjacketsmens','womensbarbourcoatsjackets','mensbarbourcoatsjackets','laurenralphlaurenouterwear','mensuperdrycoatsjackets','womensuperdrycoatsjackets', 'mensjoulescoatsjackets','womensjoulescoatsjackets','seasonscoach','seasonsmulberry','beautyelectricals','dyson','ghd','cloudnine','babyliss']

# Get themes from the table
stats_df = spark.table("ds_sandbox.next_uk_nextAds_predict_prod_complete") \
    .groupBy("theme_clean") \
    .agg(
        F.avg("repurchase_ratio").alias("rep_ratio"),
        F.sum("baskets_behavior__frequency").alias("baskets_freq")
    )
print('1')
# Calculate thresholds (distributed)
thresholds = stats_df.select(
    F.percentile_approx("rep_ratio", 0.10).alias("rep_limit"),
    F.percentile_approx("baskets_freq", 0.40).alias("freq_limit")
).collect()[0]
print('2')
# Filter and combine with your manual list
dynamic_themes_df = stats_df.filter(
    (F.col("rep_ratio") <= thresholds["rep_limit"]) & 
    (F.col("baskets_freq") >= thresholds["freq_limit"])
).select("theme_clean")
print('3')
# Convert manual list to DF and Union
manual_themes_df = spark.createDataFrame([(t,) for t in high_rep_ratio_themes_sc], ["theme_clean"])
all_penalty_themes = dynamic_themes_df.union(manual_themes_df).distinct()
display(all_penalty_themes)

In [0]:
df = full_results#spark.createDataFrame(full_results)
print('1')
# Define the Window for ranking
window_spec = Window.partitionBy("account_number").orderBy(F.col("prediction").desc())

# 1. Calculate Initial Rank
reranking_df = df.withColumn("rank", F.row_number().over(window_spec))
print('2')

# 2. Join with penalty themes (Left Join)
reranking_df = reranking_df.join(
    all_penalty_themes.withColumn("is_penalty_theme", F.lit(True)),
    reranking_df.theme == all_penalty_themes.theme_clean,
    "left"
)
print('3')

# 3. Apply Penalty Logic
penalty = 0.25
reranking_df = reranking_df.withColumn(
    "adjusted_score",
    F.when(
        (F.col("rank") == 1) & 
        (F.col("baskets_behavior__recency_rank") == 1) & 
        (F.col("is_penalty_theme") == True),
        F.col("prediction") * (1 - penalty)
    ).otherwise(F.col("prediction"))
)
print('4')

# 4. Final Re-ranking
final_window = Window.partitionBy("account_number").orderBy(F.col("adjusted_score").desc())
final_results = reranking_df.withColumn("final_rank", F.row_number().over(final_window))
print('5')

display(final_results.limit(5))

In [0]:
assert full_results.count() == final_results.count()

In [0]:
full_results = (final_results.withColumnRenamed('adjusted_score', 'ProbAggRebased')
                .withColumnRenamed('account_number', 'AccountNumber')
                .withColumnRenamed('theme', 'NextTheme')
                .withColumn('rundate', F.current_date()))
# full_results['rundate'] = pd.Timestamp.now().date()
display(full_results.limit(5))

In [0]:
# limit to top 50 ads and account_number, theme, ..{relevant cols}
# clean up themeclean > theme
theme_mapping = spark.sql("SELECT distinct theme, regexp_replace(theme, '[^a-zA-Z0-9]', '') as theme_clean FROM warehouse.next_uk_nextads_item_themes_latest where theme_rank = 1")

full_results_spark = full_results#spark.createDataFrame(full_results)
full_df_fixed = (full_results_spark.join(theme_mapping, full_results_spark['NextTheme'] == theme_mapping['theme_clean'], how='left')
                 .select('AccountNumber','theme','ProbAggRebased','rundate'))
fixed = full_df_fixed.withColumnRenamed('theme','NextTheme')
display(fixed.limit(5))

In [0]:
# validation
# check acc has more than X themes
acc_to_theme = fixed.groupBy('AccountNumber').agg(F.count_distinct('NextTheme').alias('theme_count')).filter(F.col('theme_count') < 25)
if acc_to_theme.count() > 0:
    print(f'WARN: {acc_to_theme.count()} accounts have less than 25 themes assigned')
    display(acc_to_theme)

# check there are no dupe acc:theme pairs
acc_theme_pairs = fixed.groupBy('AccountNumber', 'NextTheme').agg(F.count('*').alias('count'))
if acc_theme_pairs.filter(F.col('count') > 1).count() > 0:
    print('WARN: there are duplicate account_number:theme pairs')
    display(acc_theme_pairs.filter(F.col('count') > 1).sort(F.desc('count')))

# check all acc nums from base are present
all_accounts = (
    spark.read.table('warehouse.baskets_uk_3y')
    .filter(F.col('order_date') > F.date_sub(F.current_date(), 365))
    .select('account_number')
    .distinct()
)
if all_accounts.join(fixed.select('AccountNumber').distinct(), all_accounts['account_number'] == F.col('AccountNumber'), how='leftanti').count() > 0:
    print('ERROR: not all accounts from base are present in the model output')
    display(all_accounts.join(fixed.select('AccountNumber').distinct(), all_accounts['account_number'] == F.col('AccountNumber'), how='leftanti'))

# pull in yesterdays top 20 assigned themes, if difference > 20 message, if > 50 error
# top_themes = (
#     full_results.sort_values(['AccountNumber', 'ProbAggRebased'], ascending=[True, False])
#     .groupby('AccountNumber')
#     .first()
#     .reset_index()[['AccountNumber', 'NextTheme', 'ProbAggRebased']]
# )
# display(top_themes.groupby('NextTheme').size().reset_index(name='count').sort_values(by='count', ascending=False))

# top_themes_yest = (
#     full_results.sort_values(['AccountNumber', 'ProbAggRebased'], ascending=[True, False])
#     .groupby('AccountNumber')
#     .first()
#     .reset_index()[['AccountNumber', 'NextTheme', 'ProbAggRebased']]
# )
# display(top_themes_yest.groupby('NextTheme').size().reset_index(name='count').sort_values(by='count', ascending=False))


In [0]:
# fixed = fixed[['AccountNumber', 'NextTheme', 'ProbAggRebased', 'rundate']]
fixed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ds_sandbox.next_uk_next_ads_hackathon_model_latest")
print('written to model_latest')
# add in for the below if rundate = current_date in table, first remove these rows from the history before appending on todays
fixed.write.mode("append").saveAsTable("ds_sandbox.next_uk_next_ads_hackathon_model_full")
print('appended to model_full')